# Numerai Model Training & Submission Pipeline

This notebook is a self-contained environment for training, validating, and submitting models to Numerai. It consolidates all the logic from the `local/` scripts into a singular, top-to-bottom executable pipeline.

### Project Features
- **Data Management**: Automated download of the latest Numerai datasets.
- **Exploration**: Insights into feature distributions and target correlations.
- **Modeling**: Support for LightGBM, Neural Networks, and Transformers.
- **Validation**: Comprehensive performance metrics (CORR, MMC, Sharpe, Drawdown).
- **Submission**: Robust ensembling and pickling for Numerai model uploads.


## 1. Environment Setup
Install required dependencies for the Numerai ecosystem.


In [ ]:
!pip install -q numerapi numerai-tools lightgbm tensorflow cloudpickle pandas matplotlib scikit-learn scipy pyarrow


## 2. Configuration & Initialization
Define the data version and target settings.


In [ ]:
import json
import os
from pathlib import Path
from numerapi import NumerAPI

DATA_VERSION = 'v5.2'
napi = NumerAPI()

# Standard Numerai Target (can be modified to use auxiliary targets like target_ender_20)
MAIN_TARGET = 'target'

print(f'Using Data Version: {DATA_VERSION}')


## 3. Dataset Download
Fetch the training and validation datasets.


In [ ]:
# Download features metadata
napi.download_dataset(f'{DATA_VERSION}/features.json')
with open(f'{DATA_VERSION}/features.json') as f:
    feature_metadata = json.load(f)

feature_cols = feature_metadata['feature_sets']['medium'] # Using medium by default
target_cols = feature_metadata['targets']

# Download training and validation data (this might take a few minutes)
print('Downloading datasets...')
napi.download_dataset(f'{DATA_VERSION}/train.parquet')
napi.download_dataset(f'{DATA_VERSION}/validation.parquet')
print('Download complete.')


## 4. Exploratory Data Analysis (EDA)
Analyze target correlations and data distribution.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load a sample for EDA
train_sample = pd.read_parquet(f'{DATA_VERSION}/train.parquet', columns=['era'] + target_cols).head(100000)

# Visualize correlations
corrs = train_sample[target_cols].corrwith(train_sample[MAIN_TARGET]).sort_values(ascending=False)
corrs.head(20).plot(kind='bar', figsize=(10, 5), title=f'Top Correlations with {MAIN_TARGET}')
plt.show()

# Era distribution
train_sample.groupby('era').size().plot(title='Number of rows per era (Sample)')
plt.show()


## 5. Model Training
We will train three types of models: LightGBM, a Simple Neural Network, and a Transformer-based model.


### 5.1 LightGBM Training


In [ ]:
import lightgbm as lgb
import pickle

# Training on a subset of eras for Colab speed (optional)
train = pd.read_parquet(f'{DATA_VERSION}/train.parquet', columns=['era'] + feature_cols + [MAIN_TARGET])
train_subset = train[train['era'].isin(train['era'].unique()[::4])]

print(f'Training LightGBM on {len(train_subset)} rows...')
lgbm_model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.01,
    max_depth=5,
    num_leaves=31,
    colsample_bytree=0.1,
    verbosity=-1
)

lgbm_model.fit(
    train_subset[feature_cols],
    train_subset[MAIN_TARGET]
)

os.makedirs('models', exist_ok=True)
with open('models/lgbm_models.pkl', 'wb') as f:
    pickle.dump({MAIN_TARGET: lgbm_model}, f)
print('LightGBM model saved.')


### 5.2 Neural Network Training


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

def create_nn_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=(input_shape,)),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.1),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.1),
        layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

nn_model = create_nn_model(len(feature_cols))
print('Training Neural Network...')
nn_model.fit(train_subset[feature_cols], train_subset[MAIN_TARGET], epochs=10, batch_size=256, verbose=1)
nn_model.save('models/nn_model.keras')


### 5.3 Transformer Training


In [ ]:
class FeatureEmbedding(layers.Layer):
    def __init__(self, d_model, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
    def build(self, input_shape):
        self.projection = layers.Dense(self.d_model)
    def call(self, x):
        x = tf.expand_dims(x, axis=-1)
        return self.projection(x)
    def get_config(self):
        config = super().get_config()
        config.update({'d_model': self.d_model})
        return config

class TransformerEncoderBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ffn_dim, dropout_rate, **kwargs):
        super().__init__(**kwargs)
        self.d_model, self.num_heads, self.ffn_dim, self.dropout_rate = d_model, num_heads, ffn_dim, dropout_rate
        self.mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)
        self.ffn = models.Sequential([layers.Dense(ffn_dim, activation='relu'), layers.Dense(d_model)])
        self.layernorm1, self.layernorm2 = layers.LayerNormalization(), layers.LayerNormalization()
        self.dropout1, self.dropout2 = layers.Dropout(dropout_rate), layers.Dropout(dropout_rate)
    def call(self, x, training=False):
        attn_output = self.mha(x, x, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        x = self.layernorm1(x + attn_output)
        ffn_output = self.ffn(x, training=training)
        ffn_output = self.dropout2(ffn_output, training=training)
        x = self.layernorm2(x + ffn_output)
        return x

def create_transformer_model(num_features):
    inputs = layers.Input(shape=(num_features,))
    x = FeatureEmbedding(128)(inputs)
    pos_encoding = tf.Variable(tf.random.normal([1, num_features, 128], stddev=0.02), trainable=True)
    x = x + pos_encoding
    for _ in range(2): x = TransformerEncoderBlock(128, 4, 256, 0.1)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(1)(x)
    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='mse')
    return model

transformer_model = create_transformer_model(len(feature_cols))
print('Training Transformer...')
transformer_model.fit(train_subset[feature_cols].values.astype(np.float32), train_subset[MAIN_TARGET].values.astype(np.float32), epochs=10, batch_size=256, verbose=1)
transformer_model.save('models/transformer_model.keras')


## 6. Validation & Reporting
Evaluate the performance of our models on the validation set.


In [ ]:
from numerai_tools.scoring import numerai_corr, correlation_contribution

def compute_perf_metrics(series: pd.Series) -> dict:
    series = pd.Series(np.asarray(series).reshape(-1)).dropna()
    mean = series.mean()
    std = series.std(ddof=0)
    sharpe = mean / std if std != 0 else np.nan
    return {'mean': mean, 'std': std, 'sharpe': sharpe}

# Load validation data
val = pd.read_parquet(f'{DATA_VERSION}/validation.parquet', columns=['era', 'data_type', 'target'] + feature_cols)
val = val[val['data_type'] == 'validation']

# Generate model predictions
val['pred_lgbm'] = lgbm_model.predict(val[feature_cols])
val['pred_nn'] = nn_model.predict(val[feature_cols].values.astype(np.float32), verbose=0).reshape(-1)

# Ensemble: 75% LGBM + 25% NN (Ranked)
val['pred_ensemble'] = (0.75 * val.groupby('era')['pred_lgbm'].rank(pct=True)) + (0.25 * val.groupby('era')['pred_nn'].rank(pct=True))
val['prediction'] = val.groupby('era')['pred_ensemble'].rank(pct=True)

# Compute Metrics
per_era_corr = val.groupby('era').apply(lambda x: numerai_corr(x[['prediction']], x['target']).iloc[0])
print('Validation Metrics (CORR):')
print(pd.DataFrame({'CORR': compute_perf_metrics(per_era_corr)}))

per_era_corr.plot(kind='bar', title='Per-era CORR', figsize=(12, 5))
plt.show()


## 7. Submission Preparation
Finalize the `predict` function and save it as a pickle for Numerai upload.


In [ ]:
import cloudpickle

# Standard Numerai Submission Format
def predict(live_features: pd.DataFrame, live_benchmark_models: pd.DataFrame) -> pd.DataFrame:
    # 1. Feature Selection
    X = live_features[feature_cols]
    
    # 2. LGBM Prediction
    lgbm_pred = lgbm_model.predict(X)
    lgbm_rank = pd.Series(lgbm_pred, index=live_features.index).rank(pct=True)
    
    # 3. NN Prediction
    nn_pred = nn_model.predict(X.values.astype(np.float32), verbose=0).reshape(-1)
    nn_rank = pd.Series(nn_pred, index=live_features.index).rank(pct=True)
    
    # 4. Weighted Blend (aligned with validation)
    ensemble = (0.75 * lgbm_rank) + (0.25 * nn_rank)
    
    return ensemble.rank(pct=True, method='first').to_frame('prediction')

# Smoke Test
sample_live = val.head(100)
print('Smoke test output:')
print(predict(sample_live, None).head())

# Save for upload
with open('numerai_upload.pkl', 'wb') as f:
    f.write(cloudpickle.dumps(predict))
print('Successfully saved numerai_upload.pkl')
